# Places

Analysis of places in the individual biographies.

This notebook collects all the places mentioned in the individual biographies and uses wikidata to extract associated information about them, namely current names, coordinates, and adminstrative information (present day).

Relevant to this notebook is the list of attribute types that receive place names as values.

The list is the following:

- 'nascimento'
- 'jesuita-entrada'
- 'partida'
- 'chegada'
- 'estadia'
- 'estadia-x'
- 'jesuita-votos-local'
- 'jesuita-ordenacao-padre'
- 'morte'

Files generated by this notebook:
- "../inferences/locations_names.xlsx" persons id and name, place of stay, attribute type, wikidata, date...
- "../inferences/locations_names_places_dates_only.xlsx" without wikidata
- "../inferences/locations_names_no_wikidata.xlsx" places with no wikidata
- "inferences/wikidata-references/locations_names_wikidata.csv" 





In [4]:
from timelink.notebooks import TimelinkNotebook

tlnb = TimelinkNotebook()
tlnb.print_info()

Timelink version: 1.1.26
Project name: dehergne-repertoire
Project home: /Users/jrc/develop/dehergne-repertoire
Database type: sqlite
Database name: dehergne_repertoire
Kleio image: timelinkserver/kleio-server
Kleio server token: KAOI9...
Kleio server URL: http://127.0.0.1:8088
Kleio server home: /Users/jrc/develop/dehergne-repertoire
Kleio server container: great_greider
Kleio version requested: latest
Kleio server version: 12.9.588 (2025-06-06 16:04:22)
SQLite directory: /Users/jrc/develop/dehergne-repertoire/database/sqlite
Database version: 6ccf1ef385a6
Call print_info(show_token=True) to show the Kleio Server token
Call print_info(show_password=True) to show the Postgres password
TimelinkNotebook(project_name=dehergne-repertoire, project_home=/Users/jrc/develop/dehergne-repertoire, db_type=sqlite, db_name=dehergne_repertoire, kleio_image=timelinkserver/kleio-server, kleio_version=latest, postgres_image=postgres, postgres_version=latest)


Define attributes types that have locations as values.

In [7]:
attributes_with_place_names = [
    'nascimento',
    'jesuita-entrada',
    'partida',
    'chegada',
    'estadia',
    'estadia-x',
    'jesuita-votos-local',
    'jesuita-ordenacao-padre',
    'morte'
]

attributes_with_wikidata = [f"{attr}@wikidata" for attr in attributes_with_place_names]
attributes_with_wikidata

['nascimento@wikidata',
 'jesuita-entrada@wikidata',
 'partida@wikidata',
 'chegada@wikidata',
 'estadia@wikidata',
 'estadia-x@wikidata',
 'jesuita-votos-local@wikidata',
 'jesuita-ordenacao-padre@wikidata',
 'morte@wikidata']

Collect all the individuals with attributes related to locations. Make note of the dates of the attributes and infer dates when missing, based on last known date for a previous location attribute. Extract wikidata ids for the locations if available.



In [14]:
import pandas as pd
from timelink.pandas import entities_with_attribute
from timelink.kleio.utilities import format_timelink_date
from dehergne_util import get_linked_entity_id

# show 500 rows
pd.set_option('display.max_rows', 1550)

places_of_stay = entities_with_attribute(
    entity_type='person',
    show_elements=['name','groupname','the_line','the_order','extra_info'],
    the_type=attributes_with_place_names,
    column_name='place',
    db=tlnb.db,
)

# fillna with "?"
places_of_stay['place'] = places_of_stay['place'].fillna('?')

# remove rows where place == '?'
places_of_stay = places_of_stay[places_of_stay['place'] != '?']

# this sequence replaces missing dates with the value of the previous date
# followed by ">". This allows to use the information that although the date
# is unkown it has happened after a certain date
# we need the id in a column to group by it (it is currently in the index)
# TODO: maybe an option in entities_with_attribute and compute intervals using
#       post quem and ante quem dates. See https://github.com/time-link/timelink-kleio/issues/1

places_of_stay['id_col'] = places_of_stay.index

# We try to infer the date of stay from the previous date known
# create a copy of the date column to replace the '0' values with NaN
places_of_stay['place.date_inferred'] = places_of_stay['place.date'].replace('0', pd.NA)

# order by id and line to have the missing date values filled with the previous date
places_of_stay = places_of_stay.sort_values(by=['id_col', 'place.line'])
# use ffill to fill the missing values with the previous value
places_of_stay['place.date_inferred'] = places_of_stay.groupby(['id_col'])['place.date_inferred'].ffill()
# create a column to flag the inferred dates
places_of_stay['place.date_is_inferred'] = places_of_stay['place.date_inferred'] != places_of_stay['place.date']
# reset index
places_of_stay.reset_index(inplace=True)
# if data_is_inferred set date_inferred to date_inferrred+">"
places_of_stay.loc[places_of_stay['place.date_is_inferred'], 'place.date_inferred'] = places_of_stay['place.date_inferred'] + '>'
# set missing value to ''
places_of_stay['place.date_inferred'] = places_of_stay['place.date_inferred'].fillna('')
# restore index
places_of_stay.index = places_of_stay['id_col']

# create a new column named "wikidata_id" with the wikidata id extracted from place.comment
places_of_stay['wikidata_id'] = places_of_stay['place.comment'].apply(get_linked_entity_id, linked_data_provider='wikidata', if_missing='no wikidata')
# create a new column with the formatted date
places_of_stay['place.date_inferred.formatted'] = places_of_stay['place.date_inferred'].apply(format_timelink_date)

# places_of_stay.info()
# show results
show_only=10
cols=['place','place.type','place.attr_id','wikidata_id','place.date_inferred','place.date_inferred.formatted','groupname','name','place.comment','place.original','place.date_is_inferred','place.line','place.extra_info']
places_of_stay[places_of_stay.groupname=='n'][cols].sort_values(by=['place','place.date_inferred']).head(show_only)



,place,place.type,place.attr_id,wikidata_id,place.date_inferred,place.date_inferred.formatted,groupname,name,place.comment,place.original,place.date_is_inferred,place.line,place.extra_info
id_col,,,,,,,,,,,,,
deh-johannes-ciermans,"'s-Hertogenbosch, Holanda",nascimento,deh-johannes-ciermans-att1059-277,Q2766547,16020407,1602-04-07,n,Johannes Ciermans,@wikidata:Q2766547,Bois-le-duc,False,1507,"{'id': {'kleio_element_name': 'id', 'kleio_ele..."
deh-charles-francois-xavier-de-brevedent,Abissínia,partida,deh-charles-francois-xavier-de-brevedent-att10...,Q207521,16980610,1698-06-10,n,Charles-François Xavier de Brévedent,@wikidata:Q207521,None,False,1358,"{'the_value': {'kleio_element_name': 'valor', ..."
deh-estevao-coelho,"Abrantes, diocese da Guarda",nascimento,deh-estevao-coelho-att1198-277,Q331191,1586,1586,n,Estêvão Coelho,@wikidata:Q331191,Abrantès de Guarda?,False,1678,"{'id': {'kleio_element_name': 'id', 'kleio_ele..."
deh-antonio-sedeno,"Acapulco, México",estadia,deh-antonio-sedeno-att429-192,Q81398,15810329,1581-03-29,n,Antonio Sedeño,@wikidata:Q81398,None,False,687,"{'id': {'kleio_element_name': 'id', 'kleio_ele..."
deh-antoine-rene-de-brasle,"Acapulco, México",estadia,deh-antoine-rene-de-brasle-att990-233,Q81398,<1707,<1707,n,Antoine-René de Brasle,a caminho de Manila @wikidata:Q81398,None,False,1297,"{'the_value': {'kleio_element_name': 'valor', ..."
deh-antonio-de-magalhaes,Afeganistão,estadia,deh-antonio-de-magalhaes-att93-238,Q889,17000000,1700,n,António de Magalhães,@wikidata:Q889,None,False,145,"{'id': {'kleio_element_name': 'id', 'kleio_ele..."
deh-pierre-albier,Agen,jesuita-votos-local,deh-pierre-albier-att214-253,Q6625,16540712,1654-07-12,n,Pierre Albier,@wikidata:Q6625,None,False,294,"{'the_value': {'kleio_element_name': 'valor', ..."
deh-philippe-avril,Agen,estadia,deh-philippe-avril-att1017-253,Q6625,16910000,1691,n,Philippe Avril,@wikidata:Q6625,None,False,1442,"{'the_value': {'kleio_element_name': 'valor', ..."
deh-bento-de-gois,Agra,estadia,deh-bento-de-gois-att464-180,Q42941,16021029,1602-10-29,n,Bento de Góis,@wikidata:Q42941,None,False,614,"{'id': {'kleio_element_name': 'id', 'kleio_ele..."


### Save to Excel

In [ ]:
filename = "../inferences/locations_names.xlsx"
places_of_stay[cols].sort_values(by=['place','place.date_inferred']).to_excel(filename, sheet_name='wikidata', index=True)
# another version without wikidata_id for unbiased LLM reasoning
filename = "../inferences/locations_names_places_dates_only.xlsx"
places_of_stay[['place','place.date_inferred.formatted', 'name','place.type','place.original']].sort_values(by=['place','place.date_inferred.formatted']).to_excel(filename, sheet_name='wikidata', index=True)

### Locations with no wikidata id

In [29]:
places_of_stay[
    (places_of_stay['wikidata_id'] == 'no wikidata') &
    (~places_of_stay['place'].str.startswith('['))
][cols].sort_values(by=['place', 'place.date_inferred'])

,place,name,place.type,place.date_inferred,place.obs
id_col,,,,,
deh-jose-de-lima-ref2,Aguiar,José de Lima,nascimento,17391115,
deh-miguel-vieira,Aquilon,Miguel Vieira,estadia,,
deh-simao-da-silveira,Aquilon,Simão da Silveira,estadia,,
deh-manuel-jose-ref2,Ascitano,Emmanuel Josephus,morte,17650000,
deh-claude-francois-loppin,Bellegarde,Claude-François Loppin,nascimento,17070913,
deh-joaquim-lobo,"Cadaval, diocese de Coimbra",Joaquim Lobo,nascimento,17320224,
deh-antonio-de-gouvea,"Casal, diocese de Viseu",António de Gouveia,nascimento,1592,""""""" segundo Gouvea, A. de, & Araújo, H. P. de...."
deh-gaspar-ferreira,"Castrofornos, diocese de Viseu",Gaspar Ferreira,nascimento,1571,
deh-mauro-de-azevedo,Celorico,Mauro de Azevedo,nascimento,16470000,


#### Export to Excel all unlocated places

In [ ]:
filename = "../inferences/locations_names_no_wikidata.xlsx"
places_of_stay[places_of_stay['wikidata_id']=='no wikidata'][cols].sort_values(by=['place','place.date_inferred']).to_excel(filename, sheet_name='no_wikidata', index=True)

Export the list of wikidata ids for later processing by wikidata-linked-data notebook.

In [30]:
# save to csv wikidata column, except when value is "No wikidata"
filename = "../inferences/wikidata-references/locations_names_wikidata.csv"
places_of_stay[places_of_stay['wikidata_id'] != 'no wikidata']['wikidata_id'].drop_duplicates().to_csv(filename, index=False)

#### Only specific place of entry, unkown places

In [19]:
from timelink.pandas import entities_with_attribute

# Choose the place of entry
place_of_entry = 'Coimbra'

# get the ids of the places of entry
missionaries_from_place = entities_with_attribute(
    entity_type='person',
    show_elements=['name','groupname','the_line','the_order','extra_info'],
    the_type=['jesuita-entrada'],
    the_value=place_of_entry,
    db=tlnb.db,
)
print(f"Number of missionaries from {place_of_entry}: {len(missionaries_from_place)}")
place_of_entry_ids = missionaries_from_place.index
places_of_stay_specific = places_of_stay.loc[place_of_entry_ids]
places_of_stay_specific[places_of_stay_specific['wikidata_id'] == 'No wikidata'][cols].sort_values(by=['place','place.date_inferred'])

Number of missionaries from Coimbra: 63


,place,place.type,place.attr_id,wikidata_id,place.date_inferred,place.date_inferred.formatted,groupname,name,place.comment,place.original,place.date_is_inferred,place.line,place.extra_info
id,,,,,,,,,,,,,


In [20]:
places_of_stay_specific[places_of_stay_specific['wikidata_id'] == 'No wikidata'][cols].sort_values(by=['place','place.date_inferred']).to_excel(f"../inferences/jesuita_entrada_{place_of_entry}_places_of_stay_no_wikidata.xlsx", sheet_name='no_wikidata', index=True)

### Frequency of places / Frequência dos lugares

In [22]:
places_of_stay.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7180 entries, deh-abraham-le-royer to simao-rodrigues-ref1
Data columns (total 24 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   id                             7180 non-null   object
 1   name                           7180 non-null   object
 2   groupname                      7180 non-null   object
 3   the_line                       7180 non-null   int64 
 4   the_order                      7180 non-null   int64 
 5   extra_info                     7180 non-null   object
 6   place.attr_id                  7180 non-null   object
 7   place.type                     7180 non-null   object
 8   place                          7180 non-null   object
 9   place.date                     7180 non-null   object
 10  place.line                     7180 non-null   int64 
 11  place.level                    7180 non-null   int64 
 12  place.obs                      7

In [23]:
col = 'place' # subtotal by this column

# Use pandas groupby and specify unique value count for id
df_totals = places_of_stay.groupby(col).agg({'id':'nunique',
                                                'place.date_inferred':'min',
                                                'place.date':'max'})
df_totals.info()
df_totals.sort_values('id',ascending= False).head(130)

<class 'pandas.core.frame.DataFrame'>
Index: 1568 entries, 's-Hertogenbosch, Holanda to Žilina, Eslováquia
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   id                   1568 non-null   int64 
 1   place.date_inferred  1568 non-null   object
 2   place.date           1568 non-null   object
dtypes: int64(1), object(2)
memory usage: 49.0+ KB


,id,place.date_inferred,place.date
place,,,
Macau,495,,>1652
Pequim,255,,>17410115
Goa,209,,>1713
Cantão,200,,>16590123
China,186,,18050114
Lisboa,173,,17810300
Roma,110,,>16851112
Coimbra,90,,<16230324
Nanquim,68,,1780


In [24]:
# export totals to excel
df_totals.sort_values('id',ascending= False).to_excel(f"../inferences/locations_totals.xlsx", sheet_name='totals', index=True)

### Who was at a specific place


List those who stayed at a specific place



In [25]:
place="Coimbra"

In [26]:
cols = ['place','name','place.type','place.date_inferred','place.obs']
show_only=100
places_of_stay[places_of_stay.place==place][cols].sort_values(by=['place.date_inferred']).head(show_only)


,place,name,place.type,place.date_inferred,place.obs
id_col,,,,,
deh-nicolas-trigault,Coimbra,Nicolas Trigault,estadia-x,,
deh-bartolomeo-tedeschi,Coimbra,Bartolomeo Tedeschi,estadia-x,,
deh-belchior-miguel-carneiro-leitao,Coimbra,Belchior Miguel Carneiro Leitão,nascimento,1519,
deh-pedro-de-alcacova,Coimbra,Pedro de Alcáçova,jesuita-entrada,15420000,saiu a primeira vez
deh-pedro-martins,Coimbra,Pedro Martins,nascimento,15420000,
deh-belchior-nunes-barreto,Coimbra,Belchior Nunes Barreto,jesuita-entrada,15430311,
deh-belchior-miguel-carneiro-leitao,Coimbra,Belchior Miguel Carneiro Leitão,jesuita-entrada,15430425,
deh-francisco-perez,Coimbra,Francisco Pérez,jesuita-entrada,15440125,
deh-tiburcio-de-quadros,Coimbra,Tibúrcio de Quadros,jesuita-entrada,15440418,


#### Export to markdown

In [27]:
places_of_stay.itertuples(index=True)

In [28]:
import os
from pandas import DataFrame
from timelink.pandas import group_attributes as person_attributes

# create if it does not exist directory ../inferences/{place_of_entry}/markdown"
directory = f"../inferences/{place_of_entry}/markdown"
if not os.path.exists(directory):
    os.makedirs(directory)


for id, name in places_of_stay[places_of_stay.place==place][['name']].sort_values('name').itertuples(index=True):
    with tlnb.db.session() as session:
        p = tlnb.db.get_person(id, session=session)
        session.add(p)
        pk:str = p.to_kleio()
        pk = f"""## {name}
^[In Edit mode, select title (including the ##), right click and "Extract selection"]

---
### {name} @ timelink
```
{pk}
```

---
Select title, right click on selected title, choose extract note
"""
        fname = f"{directory}/{id}.md"
        with open(fname, 'w', encoding='utf-8') as f:
            f.write(pk)

